In [1]:
import pandas as pd
import mysql.connector
from mysql.connector import Error
import warnings
warnings.filterwarnings('ignore')
print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
connection = mysql.connector.connect(
    host='localhost',
    user='root',
    password='Keju@0573'
)
cursor = connection.cursor()
cursor.execute("CREATE DATABASE IF NOT EXISTS upi_pulse")
cursor.execute("USE upi_pulse")
print("Connected to MySQL successfully!")
print("Database 'upi_pulse' created and selected!")

Connected to MySQL successfully!
Database 'upi_pulse' created and selected!


In [3]:
df = pd.read_csv('E:\\project_upi_pulse\\data\\upi_cleaned.csv')

print("CSV loaded successfully!")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

cursor.execute("""
    CREATE TABLE IF NOT EXISTS upi_data (
        id INT AUTO_INCREMENT PRIMARY KEY,
        date DATE,
        month VARCHAR(10),
        year INT,
        upi_volume_mn FLOAT,
        upi_value_cr FLOAT,
        festival VARCHAR(50),
        is_festive INT,
        vol_mom_growth FLOAT,
        vol_yoy_growth FLOAT,
        avg_ticket_size FLOAT,
        vol_3m_avg FLOAT,
        psi_score FLOAT
    )
""")

print("Table 'upi_data' created successfully!")

CSV loaded successfully!
Shape: (83, 12)
Columns: ['date', 'month', 'year', 'upi_volume_mn', 'upi_value_cr', 'festival', 'is_festive', 'vol_mom_growth', 'vol_yoy_growth', 'avg_ticket_size', 'vol_3m_avg', 'psi_score']
Table 'upi_data' created successfully!


In [4]:
import numpy as np
df2 = df.copy()

df2 = df2.replace({np.nan: None})

insert_query = """
    INSERT INTO upi_data (
        date, month, year, upi_volume_mn, upi_value_cr, festival, is_festive, vol_mom_growth, vol_yoy_growth, avg_ticket_size, vol_3m_avg, psi_score
    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

cursor.execute("DELETE FROM upi_data")

for index, row in df2.iterrows():
    values = (
        row['date'], row['month'], row['year'], row['upi_volume_mn'], row['upi_value_cr'], row['festival'], row['is_festive'], row['vol_mom_growth'], 
        row['vol_yoy_growth'], row['avg_ticket_size'], row['vol_3m_avg'], row['psi_score']
    )
    cursor.execute(insert_query, values)

connection.commit()

print(f"Successfully inserted {len(df2)} rows into MySQL!")
print("Data is now live in upi_pulse database!")

Successfully inserted 83 rows into MySQL!
Data is now live in upi_pulse database!


In [5]:
verify_query = "SELECT COUNT(*) FROM upi_data"
cursor.execute(verify_query)

result = cursor.fetchone()

print(f"Total rows in MySQL: {result[0]}")

cursor.execute("SELECT * FROM upi_data LIMIT 5")

rows = cursor.fetchall()

columns = [desc[0] for desc in cursor.description]

df_preview = pd.DataFrame(rows, columns=columns)

print("\nFirst 5 rows from MySQL:")
df_preview

Total rows in MySQL: 83

First 5 rows from MySQL:


,id,date,month,year,upi_volume_mn,upi_value_cr,festival,is_festive,vol_mom_growth,vol_yoy_growth,avg_ticket_size,vol_3m_avg,psi_score
0,84,2019-04-01,Apr,2019,781.79,142034.0,None,0,NaN,None,1816.78,NaN,None
1,85,2019-05-01,May,2019,733.54,152449.0,None,0,-6.17173,None,2078.27,NaN,None
2,86,2019-06-01,Jun,2019,754.54,146566.0,Eid-ul-Fitr,1,2.86283,None,1942.46,756.623,None
3,87,2019-07-01,Jul,2019,822.29,146387.0,None,0,8.97898,None,1780.23,770.123,None
4,88,2019-08-01,Aug,2019,918.35,154505.0,Raksha Bandhan,1,11.68200,None,1682.42,831.727,None


In [6]:
query1 = """
    SELECT
        year,
        COUNT(*) AS total_months,
        ROUND(SUM(upi_volume_mn), 2) AS total_volume_mn,
        ROUND(AVG(upi_volume_mn), 2) AS avg_monthly_volume,
        ROUND(SUM(upi_value_cr), 2) AS total_value_cr,
        ROUND(AVG(upi_value_cr), 2) AS avg_monthly_value,
        ROUND(AVG(avg_ticket_size), 2) AS avg_ticket_size,
        ROUND(AVG(vol_mom_growth), 2) AS avg_mom_growth_pct,
        ROUND(MAX(upi_volume_mn), 2) AS peak_volume_mn,
        ROUND(MIN(upi_volume_mn), 2) AS lowest_volume_mn
    FROM upi_data
    GROUP BY year
    ORDER BY year
"""

cursor.execute(query1)

rows = cursor.fetchall()

columns = [desc[0] for desc in cursor.description]

df_yearly = pd.DataFrame(rows, columns=columns)

print("=== Query 1 - Yearly Summary ===")
df_yearly

=== Query 1 - Yearly Summary ===


,year,total_months,total_volume_mn,avg_monthly_volume,total_value_cr,avg_monthly_value,avg_ticket_size,avg_mom_growth_pct,peak_volume_mn,lowest_volume_mn
0,2019,9,8641.06,960.12,1486507.92,165167.55,1750.85,6.88,1308.40,733.54
1,2020,12,18880.89,1573.41,3387744.77,282312.06,1778.12,5.13,2234.16,999.57
2,2021,12,38744.55,3228.71,7159285.84,596607.15,1854.60,6.40,4566.30,2292.90
3,2022,12,74044.48,6170.37,12595077.81,1049589.82,1711.48,4.73,7829.49,4527.49
4,2023,12,117641.09,9803.42,18292795.25,1524399.60,1561.04,3.78,12020.23,7534.76
5,2024,12,172208.01,14350.67,24682521.12,2056876.76,1437.32,2.91,16730.01,12102.67
6,2025,12,228281.85,19023.49,29974737.50,2497894.79,1315.45,2.28,21634.67,16106.19
7,2026,2,42097.62,21048.81,5517710.50,2758855.25,1310.86,-2.86,21703.44,20394.18


In [7]:
query2 = """
    SELECT
        CASE
            WHEN is_festive = 1 THEN 'Festive Month'
            ELSE 'Non Festive Month'
        END AS month_type,
        COUNT(*) AS total_months,
        ROUND(SUM(upi_volume_mn), 2) AS total_volume_mn,
        ROUND(AVG(upi_volume_mn), 2) AS avg_volume_mn,
        ROUND(SUM(upi_value_cr), 2) AS total_value_cr,
        ROUND(AVG(upi_value_cr), 2) AS avg_value_cr,
        ROUND(AVG(avg_ticket_size), 2) AS avg_ticket_size,
        ROUND(AVG(vol_mom_growth), 2) AS avg_mom_growth
    FROM upi_data
    GROUP BY is_festive
    ORDER BY is_festive DESC
"""

cursor.execute(query2)
rows = cursor.fetchall()
columns = [desc[0] for desc in cursor.description]
df_festive = pd.DataFrame(rows, columns=columns)

print("=== Query 2 - Festive vs Non Festive ===")
df_festive

=== Query 2 - Festive vs Non Festive ===


,month_type,total_months,total_volume_mn,avg_volume_mn,total_value_cr,avg_value_cr,avg_ticket_size,avg_mom_growth
0,Festive Month,39,315455.02,8088.59,46795145.56,1199875.53,1623.85,7.35
1,Non Festive Month,44,385084.53,8751.92,56301235.16,1279573.53,1612.39,1.53


In [8]:
query3 = """
    SELECT
        month,
        year,
        ROUND(upi_volume_mn, 2) AS upi_volume_mn,
        ROUND(upi_value_cr, 2) AS upi_value_cr,
        ROUND(avg_ticket_size, 2) AS avg_ticket_size,
        ROUND(vol_mom_growth, 2) AS vol_mom_growth,
        COALESCE(festival, 'No Festival') AS festival
    FROM upi_data
    WHERE vol_mom_growth IS NOT NULL
    ORDER BY vol_mom_growth DESC
    LIMIT 10
"""

cursor.execute(query3)
rows = cursor.fetchall()
columns = [desc[0] for desc in cursor.description]
df_spikes = pd.DataFrame(rows, columns=columns)

print("=== Query 3 - Top 10 Spike Months ===")
df_spikes

=== Query 3 - Top 10 Spike Months ===


,month,year,upi_volume_mn,upi_value_cr,avg_ticket_size,vol_mom_growth,festival
0,May,2020,1234.50,218391.59,1769.07,23.50,Eid-ul-Fitr
1,Oct,2019,1148.36,191359.94,1666.38,20.24,Diwali
2,Mar,2022,5405.65,960581.69,1777.00,19.40,Holi
3,Mar,2021,2731.68,504886.44,1848.26,19.14,Holi
4,Jul,2021,3247.82,606281.12,1866.73,15.68,No Festival
5,Oct,2021,4218.65,771445.00,1828.65,15.44,Navaratri
6,Mar,2023,8685.30,1410443.00,1623.94,15.27,Holi
7,Oct,2020,2071.62,386106.75,1863.79,15.08,Navaratri
8,Mar,2025,18301.51,2477221.50,1353.56,13.63,Holi
9,Jul,2020,1497.36,290537.88,1940.33,12.00,No Festival


In [9]:
query4 = """
    SELECT
        CASE
            WHEN month IN ('Apr', 'May', 'Jun')
                THEN 'Q1 (Apr-Jun)'
            WHEN month IN ('Jul', 'Aug', 'Sep')
                THEN 'Q2 (Jul-Sep)'
            WHEN month IN ('Oct', 'Nov', 'Dec')
                THEN 'Q3 (Oct-Dec)'
            WHEN month IN ('Jan', 'Feb', 'Mar')
                THEN 'Q4 (Jan-Mar)'
        END AS fiscal_quarter,
        COUNT(*) AS total_months,
        ROUND(SUM(upi_volume_mn), 2) AS total_volume_mn,
        ROUND(AVG(upi_volume_mn), 2) AS avg_volume_mn,
        ROUND(SUM(upi_value_cr), 2) AS total_value_cr,
        ROUND(AVG(upi_value_cr), 2) AS avg_value_cr,
        ROUND(AVG(avg_ticket_size), 2) AS avg_ticket_size,
        ROUND(AVG(psi_score), 2) AS avg_psi_score
    FROM upi_data
    GROUP BY fiscal_quarter
    ORDER BY avg_volume_mn DESC
"""

cursor.execute(query4)
rows = cursor.fetchall()
columns = [desc[0] for desc in cursor.description]
df_quarterly = pd.DataFrame(rows, columns=columns)

print("=== Query 4 - Quarterly Seasonality ===")
df_quarterly

=== Query 4 - Quarterly Seasonality ===


,fiscal_quarter,total_months,total_volume_mn,avg_volume_mn,total_value_cr,avg_value_cr,avg_ticket_size,avg_psi_score
0,Q3 (Oct-Dec),21,191871.21,9136.72,28091655.80,1337697.90,1586.94,61.35
1,Q4 (Jan-Mar),20,181259.11,9062.96,26757544.73,1337877.24,1603.14,53.59
2,Q2 (Jul-Sep),21,172374.85,8208.33,24894130.00,1185434.76,1604.23,74.22
3,Q1 (Apr-Jun),21,155034.38,7382.59,23353050.19,1112050.01,1676.10,60.26


In [11]:
query5 = """
    SELECT
        month,
        year,
        ROUND(psi_score, 2) AS psi_score,
        ROUND(upi_volume_mn, 2) AS upi_volume_mn,
        ROUND(avg_ticket_size, 2) AS avg_ticket_size,
        ROUND(vol_mom_growth, 2) AS vol_mom_growth,
        COALESCE(festival, 'No Festival') AS festival,
        CASE
            WHEN psi_score >= 75 THEN 'High Stress'
            WHEN psi_score >= 50 THEN 'Moderate Stress'
            WHEN psi_score >= 25 THEN 'Low Stress'
            ELSE 'Minimal Stress'
        END AS stress_level
    FROM upi_data
    WHERE psi_score IS NOT NULL
    ORDER BY psi_score DESC
    LIMIT 15
"""

cursor.execute(query5)
rows = cursor.fetchall()
df_stress = pd.DataFrame(rows, columns=columns)

print("=== Query 5 - PSI Stress Periods ===")
df_stress

=== Query 5 - PSI Stress Periods ===


,fiscal_quarter,total_months,total_volume_mn,avg_volume_mn,total_value_cr,avg_value_cr,avg_ticket_size,avg_psi_score
0,Mar,2022,100.00,5405.65,1777.00,19.40,Holi,High Stress
1,Mar,2024,94.40,13440.00,1471.99,11.05,Holi,High Stress
2,Aug,2024,93.69,14963.05,1377.22,3.65,Raksha Bandhan,High Stress
3,Mar,2025,92.92,18301.51,1353.56,13.63,Holi,High Stress
4,Aug,2025,92.03,20008.31,1242.22,2.78,Raksha Bandhan,High Stress
5,Aug,2023,91.74,10586.02,1489.26,6.24,Raksha Bandhan,High Stress
6,Apr,2023,90.66,8863.26,1597.05,2.05,Eid-ul-Fitr,High Stress
7,Aug,2022,90.07,6579.63,1630.48,4.63,Raksha Bandhan,High Stress
8,Aug,2021,87.74,3555.55,1797.52,9.47,Raksha Bandhan,High Stress
9,Apr,2022,84.77,5583.05,1761.23,3.28,No Festival,High Stress


In [12]:
cursor.execute(query5)
rows = cursor.fetchall()
columns = [desc[0] for desc in cursor.description]
df_stress = pd.DataFrame(rows, columns=columns)

print("=== Query 5 - PSI Stress Periods ===")
print(df_stress.columns.tolist())
df_stress

=== Query 5 - PSI Stress Periods ===
['month', 'year', 'psi_score', 'upi_volume_mn', 'avg_ticket_size', 'vol_mom_growth', 'festival', 'stress_level']


,month,year,psi_score,upi_volume_mn,avg_ticket_size,vol_mom_growth,festival,stress_level
0,Mar,2022,100.00,5405.65,1777.00,19.40,Holi,High Stress
1,Mar,2024,94.40,13440.00,1471.99,11.05,Holi,High Stress
2,Aug,2024,93.69,14963.05,1377.22,3.65,Raksha Bandhan,High Stress
3,Mar,2025,92.92,18301.51,1353.56,13.63,Holi,High Stress
4,Aug,2025,92.03,20008.31,1242.22,2.78,Raksha Bandhan,High Stress
5,Aug,2023,91.74,10586.02,1489.26,6.24,Raksha Bandhan,High Stress
6,Apr,2023,90.66,8863.26,1597.05,2.05,Eid-ul-Fitr,High Stress
7,Aug,2022,90.07,6579.63,1630.48,4.63,Raksha Bandhan,High Stress
8,Aug,2021,87.74,3555.55,1797.52,9.47,Raksha Bandhan,High Stress
9,Apr,2022,84.77,5583.05,1761.23,3.28,No Festival,High Stress


In [13]:
import os

output_path = 'E:\\project_upi_pulse\\data\\sql_results\\'
os.makedirs(output_path, exist_ok=True)

df_yearly.to_csv(output_path + 'yearly_summary.csv', index=False)

df_festive.to_csv(output_path + 'festive_comparison.csv', index=False)

df_spikes.to_csv(output_path + 'top10_spikes.csv', index=False)

df_quarterly.to_csv(output_path + 'quarterly_seasonality.csv', index=False)

df_stress.to_csv(output_path + 'psi_stress_periods.csv', index=False)

print("All query results exported successfully!")
print(f"\nFiles saved in: {output_path}")
print("\nFiles created:")
print("1. yearly_summary.csv")
print("2. festive_comparison.csv")
print("3. top10_spikes.csv")
print("4. quarterly_seasonality.csv")
print("5. psi_stress_periods.csv")

connection.close()

print("\nMySQL connection closed successfully!")

All query results exported successfully!

Files saved in: E:\project_upi_pulse\data\sql_results\

Files created:
1. yearly_summary.csv
2. festive_comparison.csv
3. top10_spikes.csv
4. quarterly_seasonality.csv
5. psi_stress_periods.csv

MySQL connection closed successfully!
